# سبق 03 - ایجنٹک ڈیزائن پیٹرنز

اس سبق میں، ہم موثر AI ایجنٹس بنانے کے لیے تین بنیادی ڈیزائن پیٹرنز کو دریافت کرتے ہیں:

1. **واضح ایجنٹ ہدایات** — ٹھوس، کردار کی وضاحت کرنے والے پرامپس بنانا جو ایجنٹ کے رویے کو راہنمائی فراہم کریں  
2. **Pydantic ماڈلز کے ساتھ ساختہ آؤٹ پٹ** — اس بات کو یقینی بنانا کہ ایجنٹس پیش گوئی کے قابل، مصدقہ ڈیٹا واپس کریں  
3. **واحد ذمہ داری ایجنٹس** — مرکوز ایجنٹس ڈیزائن کرنا جو ہر ایک کام کو اچھی طرح انجام دے

ہم ہر پیٹرن کو ایک **سفر کی منزل تجویز کرنے والے** منظر نامے پر لاگو کریں گے، تدریجی طور پر ایک ایسا نظام تیار کرتے ہوئے جو منزلیں تجویز کر سکے، دستیابی چیک کر سکے، اور لاجسٹکس کو سنبھال سکے۔


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## پیٹرن 1: ایجنٹ کے لیے واضح ہدایات

سب سے مؤثر پیٹرن سب سے آسان بھی ہے: اپنے ایجنٹ کے لیے واضح، تفصیلی ہدایات لکھنا۔

اچھی ہدایات یہ تعین کرتی ہیں:
- **کون** ایجنٹ ہے (شخصیت اور انداز)
- **کیا** اسے کرنا چاہیے (قدم بہ قدم ذمہ داریاں)
- **کیسے** اس کا رویہ ہونا چاہیے (محدودیات اور انداز)

نیچے، ہم ایک سفری کنسیئر ایجنٹ تخلیق کرتے ہیں جس کے واضح ہدایات ہر جواب کو شکل دیتی ہیں جو یہ تیار کرتا ہے۔


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## پیٹرن 2: پائیڈینٹک ماڈلز کے ساتھ منظم آؤٹ پٹ

آزادانہ متن گفتگو کے لیے مفید ہے، لیکن نیچے کے نظاموں کو منظم ڈیٹا کی ضرورت ہوتی ہے۔
**پائیڈینٹک ماڈلز** کو ایک **ٹول فنکشن** کے ساتھ جوڑ کر، ہم یہ کر سکتے ہیں:

- ایجنٹ کے آؤٹ پٹ کے لیے ایک درست اسکیمہ متعین کریں
- جوابات کو خودکار طور پر درست کریں
- ایجنٹ کے نتائج کو قابل اعتماد طریقے سے ایپلیکیشن لاجک میں شامل کریں

ہم ایک ایسا ٹول بھی متعارف کراتے ہیں جو منزل کی تفصیلات واپس کرتا ہے تاکہ ایجنٹ اپنی سفارشات کو حقیقی ڈیٹا کی بنیاد پر رکھ سکے۔


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## پیٹرن 3: واحد ذمہ داری والے ایجنٹس

پیچیدہ کاموں کا فائدہ یہ ہوتا ہے کہ ان کو کئی مخصوص ایجنٹس میں تقسیم کیا جائے، جن میں سے ہر ایک کی ایک واحد ذمہ داری ہو:

- ایک **منزل کا ماہر** جو جگہوں اور دستیابی کے بارے میں جانتا ہو
- ایک **لاجسٹکس منصوبہ ساز** جو پروازوں، ہوٹلوں، اور سفرناموں کو سنبھالے

یہ سافٹ ویئر انجینئرنگ کے اصول *تفویض ذمہ داری* کی عکاسی کرتا ہے — ہر ایجنٹ کو علیحدہ طور پر آسانی سے آزمایا، سنبھالا، اور بہتر بنایا جا سکتا ہے۔


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصہ

اس سبق میں ہم نے ایک سفری سفارش کنندہ منظر نامے پر تین ایجنٹک ڈیزائن پیٹرنز لاگو کیے:

| پیٹرن | کلیدی خیال | فائدہ |
|---|---|---|
| **واضح ہدایات** | شخصیت، ذمہ داریاں، اور پابندیاں پہلے سے متعین کریں | مستقل، برانڈ کے مطابق ایجنٹ کا رویہ |
| **محدد آؤٹ پٹ** | جوابی فارمیٹ کے طور پر Pydantic ماڈلز استعمال کریں | درست، مشین قابلِ فہم نتائج |
| **واحد ذمہ داری** | ہر ایجنٹ کو ایک خاص کام دیں | جانچ، دیکھ بھال، اور ترکیب آسان ہو جاتی ہے |

یہ پیٹرنز قدرتی طور پر ملاپ پذیر ہیں — آپ واضح ہدایات کو محدد آؤٹ پٹ کے ساتھ ایک واحد ذمہ داری والے ایجنٹ میں ملا کر مضبوط، تیار برائے پیداوار نظام بنا سکتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
